In [ ]:
import os

import random
import json
from typing import List, Dict, Any

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, LSTM, Dense, Dropout, BatchNormalization, Concatenate, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [ ]:
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
tf.config.experimental.enable_op_determinism()

In [ ]:
# === Load and filter stock data ===
stocks = ['ENB','GS','WFC','GME','D','EA','CMCSA','DHI','CRM','VRTX',
          'SPWR','GILD','WDC','BX','AAL']

df = (
    pd.read_csv('fnspid_prices_title_sentiment.csv',
                parse_dates=['date'],
                index_col='date')
      .query("Stock_symbol in @stocks")
)

# === Generate binary target ===
def make_target(group):
    group['target_binary'] = (group['adj close'].shift(-1) > group['adj close']).astype(int)
    return group

df = (
    df.groupby('Stock_symbol', group_keys=False)
      .apply(make_target)
      .dropna(subset=['target_binary'])
)

# === Helper to load and daily-resample macro data ===
def load_macro(path, date_col='observation_date'):
    return (
        pd.read_csv(path, parse_dates=[date_col], index_col=date_col)
          .resample('D')
          .fillna(method='ffill').fillna(method='bfill')
    )

# === Load and combine all macro series ===
macro_paths = {
    'DFF': 'interest_rates.csv',
    'CPIAUCSL': 'cpi.csv',
    'UNRATE': 'unemployment_rate.csv',
    'PPI': 'ppi.csv',
    'GOLD_OIL_RATES': 'gold_silver_rates_oil_1999_2024.csv',
    'DTWEXBGS': 'nominal_us_dollar_index.csv'
}
macro_dfs = [load_macro(p) for p in macro_paths.values()]
merged_macro = (
    pd.concat(macro_dfs, axis=1)
      .reindex(df.index.unique())  # match only stock dates
      .fillna(method='ffill')      # forward-fill missing macro values
      .fillna(method='bfill')      # backfill start gaps if any
)

# === Join stock data with macro data ===
merged = df.join(merged_macro, how='left')
merged = merged.drop(columns=['Vol._x', 'Vol._y'])
# === Check ===
print(merged.info())
print(merged.head())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 33660 entries, 2015-01-05 to 2023-12-01
Data columns (total 39 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   volume                           33660 non-null  float64
 1   open                             33660 non-null  float64
 2   high                             33660 non-null  float64
 3   low                              33660 non-null  float64
 4   close                            33660 non-null  float64
 5   adj close                        33660 non-null  float64
 6   Stock_symbol                     33660 non-null  object 
 7   avg_weighted_sent                33660 non-null  float64
 8   avg_score                        33660 non-null  float64
 9   article_count                    33660 non-null  float64
 10  movement_percent                 33660 non-null  float64
 11  target_binary                    33660 non-null  int64  
 12  D

/tmp/ipython-input-634012813.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(make_target)
/tmp/ipython-input-634012813.py:28: FutureWarning: DatetimeIndexResampler.fillna is deprecated and will be removed in a future version. Use obj.ffill(), obj.bfill(), or obj.nearest() instead.
  .fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-634012813.py:28: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-634012813.py:28: FutureWarning: DatetimeIndexResampler.fillna is deprecated and will be removed in a future version. Use o

In [ ]:
# ======================
# Reproducibility
# ======================
def set_seed(seed: int):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# ======================
# Sequence generator
# ======================
def make_sequences_dual(df, window_size, price_cols, sent_cols, target, skip_all_zero_sent=True):
    X_price, X_sent, y = [], [], []
    for i in range(len(df) - window_size):
        price_seq = df.iloc[i:i+window_size][price_cols].values
        sent_seq = df.iloc[i:i+window_size][sent_cols].values
        if skip_all_zero_sent and np.all(sent_seq == 0):
            continue
        target_val = df.iloc[i+window_size][target]
        X_price.append(price_seq)
        X_sent.append(sent_seq)
        y.append(target_val)
    return np.array(X_price), np.array(X_sent), np.array(y)

# ======================
# Temporal Attention Layer
# ======================
class TemporalAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(shape=(input_shape[-1],),
                                 initializer='zeros', trainable=True)
        self.u = self.add_weight(shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        super().build(input_shape)

    def call(self, x):
        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        ait = tf.nn.softmax(tf.tensordot(uit, self.u, axes=1), axis=1)
        out = tf.reduce_sum(x * ait, axis=1)
        return out

# ======================
# Feature Attention Layer
# ======================
class FeatureAttention(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]),
                                 initializer='glorot_uniform',
                                 trainable=True)
        self.b = self.add_weight(shape=(input_shape[-1],),
                                 initializer='zeros', trainable=True)
        self.u = self.add_weight(shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform',
                                 trainable=True)
        super().build(input_shape)

    def call(self, x):
        # x: (batch, timesteps, features)
        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)  # (batch, timesteps, features)
        ait = tf.nn.softmax(tf.tensordot(uit, self.u, axes=1), axis=-1)  # attention over features
        out = x * ait
        return out

# ======================
# Model builder with feature attention
# ======================
def build_model_from_config(window_size, price_n_features, sent_n_features, config):
    l2_value = config.get('l2_value', 1e-4)
    learning_rate = config.get('learning_rate', 1e-4)

    # Price branch
    price_in = Input(shape=(window_size, price_n_features), name='price_input')
    x = price_in

    # Apply CNN / BatchNorm / Dropout layers before LSTM
    for layer_cfg in config.get('price_stack', []):
        t = layer_cfg['type'].lower()
        if t == 'conv1d':
            x = Conv1D(filters=layer_cfg.get('filters', 16),
                       kernel_size=layer_cfg.get('kernel_size', 3),
                       activation=layer_cfg.get('activation', 'relu'),
                       padding=layer_cfg.get('padding', 'same'))(x)
        elif t == 'batchnorm':
            x = BatchNormalization()(x)
        elif t == 'dropout':
            x = Dropout(rate=layer_cfg.get('rate', 0.2))(x)
        elif t == 'lstm':
            break  # stop before LSTM

    # Feature attention before LSTM if enabled
    if config.get('feature_attention', False):
        x = FeatureAttention()(x)

    # Apply LSTM layers from price_stack
    lstm_layers = [l for l in config.get('price_stack', []) if l['type'].lower() == 'lstm']
    for i, layer_cfg in enumerate(lstm_layers):
        # Return sequences True for all except last
        return_seq = layer_cfg.get('return_sequences', True)
        if i == len(lstm_layers) - 1:
            return_seq = False  # final LSTM outputs (batch, features) for merging
        x = LSTM(layer_cfg.get('units', 32),
                 recurrent_dropout=layer_cfg.get('recurrent_dropout', 0.0),
                 return_sequences=return_seq)(x)

    # Sentiment branch
    sent_in = Input(shape=(window_size, sent_n_features), name='sent_input')
    s = sent_in
    for layer_cfg in config.get('sent_stack', []):
        t = layer_cfg['type'].lower()
        if t == 'lstm':
            s = LSTM(layer_cfg.get('units', 8),
                     recurrent_dropout=layer_cfg.get('recurrent_dropout', 0.0),
                     return_sequences=layer_cfg.get('return_sequences', True))(s)
        elif t == 'dropout':
            s = Dropout(rate=layer_cfg.get('rate', 0.2))(s)
        elif t == 'batchnorm':
            s = BatchNormalization()(s)

    # Apply temporal attention to sent branch
    if config.get('sent_attention', True):
        s = TemporalAttention()(s)
    else:
        s = LSTM(config.get('sent_final_lstm_units', 8), return_sequences=False)(s)

    # Merge and output
    merged = Concatenate()([x, s])
    out = Dense(1, activation='sigmoid', kernel_regularizer=l2(l2_value))(merged)

    model = Model(inputs=[price_in, sent_in], outputs=out)
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['AUC', 'accuracy'])
    return model



In [ ]:
def overlapping_fixed_tscv(n_samples, n_splits, train_size, val_size, test_size):
    """
    Fixed-size overlapping rolling time series splits.
    Each fold shifts forward by step = test_size.
    """
    total_window = train_size + val_size + test_size
    step = test_size  # overlap controlled by test window

    for i in range(n_splits):
        start = i * step
        end = start + total_window
        if end > n_samples:
            break
        train_idx = np.arange(start, start + train_size)
        val_idx = np.arange(start + train_size, start + train_size + val_size)
        test_idx = np.arange(start + train_size + val_size, start + total_window)
        yield train_idx, val_idx, test_idx

# ======================
# Training loop (fixed)
# ======================
def train_on_merged(merged, config):
    set_seed(config.get('seed', 0))
    window_size = config['window_size']
    price_cols = config['price_cols']
    macro_cols = config['macro_cols']
    sent_cols = config['sent_cols']
    target = config['target']
    feature_cols = config['feature_cols'] or [f"{c}_logret" for c in price_cols] + [f"{c}_ret" for c in macro_cols]

    results = []

    for symbol in merged['Stock_symbol'].unique():
        print(f"\n==== Training for: {symbol} ====")
        df_symbol = merged[merged['Stock_symbol'] == symbol].copy()

        # Compute returns
        for col in price_cols:
            safe = df_symbol[col].replace(0, np.nan)
            df_symbol[f'{col}_logret'] = np.log(safe) - np.log(safe.shift(1))
        for col in macro_cols:
            df_symbol[f'{col}_ret'] = df_symbol[col].pct_change()
        df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')

        Xp_full, Xs_full, y_full = make_sequences_dual(df_symbol, window_size, feature_cols, sent_cols, target)
        if len(Xp_full) == 0:
            print(f"No sequences for {symbol}, skipping.")
            continue

        X_idx = np.arange(len(Xp_full))

        fold = 0
        for train_idx, val_idx, test_idx in overlapping_fixed_tscv(
                n_samples=len(X_idx),
                n_splits=5,
                train_size=1000,
                val_size=150,
                test_size=150):

            fold += 1
            print(f"\n-- Fold {fold} --")
            Xp_train, Xp_val, Xp_test = Xp_full[train_idx], Xp_full[val_idx], Xp_full[test_idx]
            Xs_train, Xs_val, Xs_test = Xs_full[train_idx], Xs_full[val_idx], Xs_full[test_idx]
            y_train, y_val, y_test = y_full[train_idx], y_full[val_idx], y_full[test_idx]

            def dist(name, y):
                ones = np.sum(y == 1)
                zeros = np.sum(y == 0)
                print(f"{name} -> 1: {ones/len(y)*100:.2f}% | 0: {zeros/len(y)*100:.2f}% (n={len(y)})")

            dist("Train", y_train)
            dist("Validation", y_val)
            dist("Test", y_test)

            model = build_model_from_config(window_size, len(feature_cols), len(sent_cols), config)
            early_stop = EarlyStopping(
                monitor='val_loss',
                patience=config['early_stopping_patience'],
                restore_best_weights=True
            )

            model.fit(
                [Xp_train, Xs_train], y_train,
                validation_data=([Xp_val, Xs_val], y_val),
                epochs=config['epochs'],
                batch_size=config['batch_size'],
                callbacks=[early_stop],
                verbose=0
            )

            y_prob = model.predict([Xp_test, Xs_test])
            y_pred = (y_prob > 0.5).astype(int)

            auc = roc_auc_score(y_test, y_prob)
            f1 = f1_score(y_test, y_pred)
            mcc = matthews_corrcoef(y_test, y_pred)
            acc = np.mean(y_pred.flatten() == y_test.flatten())
            cm = confusion_matrix(y_test, y_pred)

            print(f"[{symbol}][Fold {fold}] AUC: {auc:.4f} | F1: {f1:.4f} | MCC: {mcc:.4f} | ACC: {acc:.4f}")
            print(f"Confusion matrix:\n{cm}\n")

            results.append({
                'Symbol': symbol,
                'Fold': fold,
                'Test_AUC': auc,
                'Test_F1': f1,
                'Test_MCC': mcc,
                'Test_ACC': acc
            })

    results_df = pd.DataFrame(results)
    return results_df

In [ ]:
def summarize_results(results_df):
    """Summarize per-stock and overall mean AUC and ACC."""
    grouped = results_df.groupby('Symbol')

    acc_by_symbol = grouped['Test_ACC'].mean().sort_values(ascending=False)
    auc_by_symbol = grouped['Test_AUC'].mean().sort_values(ascending=False)
    mcc_by_symbol = grouped['Test_MCC'].mean().sort_values(ascending=False)
    f1_by_symbol = grouped['Test_F1'].mean().sort_values(ascending=False)


    print("=== Per-Stock Summary ===")
    print("\nMean Test Accuracy by Symbol:")
    print(acc_by_symbol)

    print("\nMean Test AUC by Symbol:")
    print(auc_by_symbol)

    print("\nMean Test MCC by Symbol:")
    print(mcc_by_symbol)

    print("\nMean Test F1 by Symbol:")
    print(f1_by_symbol)

    print("\n=== Overall Summary ===")
    print(f"Overall Mean ACC: {acc_by_symbol.mean():.4f}")
    print(f"Overall Mean AUC: {auc_by_symbol.mean():.4f}")
    print(f"Overall Mean MCC: {mcc_by_symbol.mean():.4f}")
    print(f"Overall Mean F1: {f1_by_symbol.mean():.4f}")

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 10,
    "price_cols": ['open', 'high', 'low', 'close', 'volume'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}

# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 196ms/step
[AAL][Fold 1] AUC: 0.5094 | F1: 0.6186 | MCC: 0.0493 | ACC: 0.5067
Confusion matrix:
[[16 62]
 [12 60]]


-- Fold 2 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 187ms/step
[AAL][Fold 2] AUC: 0.5027 | F1: 0.1895 | MCC: -0.1129 | ACC: 0.4867
Confusion matrix:
[[64 18]
 [59  9]]


-- Fold 3 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 178ms/step
[AAL][Fold 3] AUC: 0.4494 | F1: 0.6250 | MCC: -0.0311 | ACC: 0.4800
Confusion matrix:
[[ 7 70]
 [ 8 65]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.50% | 0: 49.50% (n=1000)
Validation -> 1: 56.67% | 0: 43.33% (n=150)
Test -> 1: 58.00% | 0: 42.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 178ms/step
[BX][Fold 1] AUC: 0.4348 | F1: 0.6169 | MCC: -0.1303 | ACC: 0.4867
Confusion matrix:
[[11 52]
 [25 62]]


-- Fold 2 --
Train -> 1: 51.50% | 0: 48.50% (n=1000)
Validation -> 1: 58.00% | 0: 42.00% (n=150)
Test -> 1: 57.33% | 0: 42.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 176ms/step
[BX][Fold 2] AUC: 0.4878 | F1: 0.7193 | MCC: 0.0352 | ACC: 0.5733
Confusion matrix:
[[ 4 60]
 [ 4 82]]


-- Fold 3 --
Train -> 1: 53.30% | 0: 46.70% (n=1000)
Validation -> 1: 57.33% | 0: 42.67% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step
[BX][Fold 3] AUC: 0.6011 | F1: 0.7037 | MCC: 0.1592 | ACC: 0.5733
Confusion matrix:
[[10 60]
 [ 4 76]]


-- Fold 4 --
Train -> 1: 55.20% | 0: 44.80% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1

...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 177ms/step
[VRTX][Fold 1] AUC: 0.5001 | F1: 0.6937 | MCC: 0.0484 | ACC: 0.5467
Confusion matrix:
[[ 5 64]
 [ 4 77]]


-- Fold 2 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 175ms/step
[VRTX][Fold 2] AUC: 0.5114 | F1: 0.5375 | MCC: 0.0118 | ACC: 0.5067
Confusion matrix:
[[33 41]
 [33 43]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 171ms/step
[VRTX][Fold 3] AUC: 0.4922 | F1: 0.4706 | MCC: -0.0711 | ACC: 0.4600
Confusion matrix:
[[33 48]
 [33 36]]


-- Fold 4 --
Train -> 1: 52.10% | 0: 47.90% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.70% | 0: 49.30% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 18s 4s/step
[WDC][Fold 1] AUC: 0.5096 | F1: 0.4186 | MCC: 0.0135 | ACC: 0.5000
Confusion matrix:
[[48 24]
 [51 27]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 254ms/step
[WDC][Fold 2] AUC: 0.5679 | F1: 0.5595 | MCC: 0.0412 | ACC: 0.5067
Confusion matrix:
[[29 52]
 [22 47]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 51.33% | 0: 48.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 286ms/step
[WDC][Fold 3] AUC: 0.4805 | F1: 0.5157 | MCC: -0.0293 | ACC: 0.4867
Confusion matrix:
[[32 41]
 [36 41]]


-- Fold 4 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 51.33% | 0: 48.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 185ms/step
[WFC][Fold 1] AUC: 0.4400 | F1: 0.3115 | MCC: -0.1415 | ACC: 0.4400
Confusion matrix:
[[47 31]
 [53 19]]


-- Fold 2 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 192ms/step
[WFC][Fold 2] AUC: 0.5012 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.4933
Confusion matrix:
[[74  0]
 [76  0]]


-- Fold 3 --
Train -> 1: 49.10% | 0: 50.90% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 272ms/step
[WFC][Fold 3] AUC: 0.4901 | F1: 0.5122 | MCC: -0.0759 | ACC: 0.4667
Confusion matrix:
[[28 41]
 [39 42]]


-- Fold 4 --
Train -> 1: 48.70% | 0: 51.30% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 10,
    "price_cols": ['open', 'high', 'low', 'close', 'volume'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 363ms/step
[AAL][Fold 1] AUC: 0.5518 | F1: 0.6481 | MCC: 0.0599 | ACC: 0.4933
Confusion matrix:
[[ 4 74]
 [ 2 70]]


-- Fold 2 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 250ms/step
[AAL][Fold 2] AUC: 0.5025 | F1: 0.5608 | MCC: -0.0628 | ACC: 0.4467
Confusion matrix:
[[14 68]
 [15 53]]


-- Fold 3 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)


1/5 ━━━━━━━━━━━━━━━━━━━━ 3s 936ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 250ms/step
[AAL][Fold 3] AUC: 0.4800 | F1: 0.3448 | MCC: -0.0273 | ACC: 0.4933
Confusion matrix:
[[54 23]
 [53 20]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 249ms/step
[AAL][Fold 4] AUC: 0.4464 | F1: 0.4968 | MCC: -0.0433 | ACC: 0.4733
Confusion matrix:
[[32 48]
 [31 39]]


-- Fold 5 --
Train -> 1: 47.40% | 0: 52.60% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 258ms/step
[AAL][Fold 5] AUC: 0.5170 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.5333
Confusion matrix:
[[80  0]
 [70  0]]


==== Training for: BX ====


...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 251ms/step
[VRTX][Fold 1] AUC: 0.4382 | F1: 0.7013 | MCC: 0.0000 | ACC: 0.5400
Confusion matrix:
[[ 0 69]
 [ 0 81]]


-- Fold 2 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 251ms/step
[VRTX][Fold 2] AUC: 0.5165 | F1: 0.5175 | MCC: 0.0819 | ACC: 0.5400
Confusion matrix:
[[44 30]
 [39 37]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 267ms/step
[VRTX][Fold 3] AUC: 0.4815 | F1: 0.5269 | MCC: -0.0304 | ACC: 0.4733
Confusion matrix:
[[27 54]
 [25 44]]


-- Fold 4 --
Train -> 1: 52.10% | 0: 47.90% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.70% | 0: 49.30% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 343ms/step
[WDC][Fold 1] AUC: 0.5034 | F1: 0.2000 | MCC: -0.0543 | ACC: 0.4667
Confusion matrix:
[[60 12]
 [68 10]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 475ms/step
[WDC][Fold 2] AUC: 0.5534 | F1: 0.5730 | MCC: 0.0258 | ACC: 0.4933
Confusion matrix:
[[23 58]
 [18 51]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 51.33% | 0: 48.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 247ms/step
[WDC][Fold 3] AUC: 0.5424 | F1: 0.6471 | MCC: 0.0299 | ACC: 0.5200
Confusion matrix:
[[12 61]
 [11 66]]


-- Fold 4 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 51.33% | 0: 48.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 245ms/step
[WFC][Fold 1] AUC: 0.4516 | F1: 0.4493 | MCC: -0.0183 | ACC: 0.4933
Confusion matrix:
[[43 35]
 [41 31]]


-- Fold 2 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 247ms/step
[WFC][Fold 2] AUC: 0.4806 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.4933
Confusion matrix:
[[74  0]
 [76  0]]


-- Fold 3 --
Train -> 1: 49.10% | 0: 50.90% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 249ms/step
[WFC][Fold 3] AUC: 0.4550 | F1: 0.6154 | MCC: -0.0489 | ACC: 0.5000
Confusion matrix:
[[15 54]
 [21 60]]


-- Fold 4 --
Train -> 1: 48.70% | 0: 51.30% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 10,
    "price_cols": ['open', 'high', 'low', 'close', 'volume'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}

# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 351ms/step
[AAL][Fold 1] AUC: 0.5267 | F1: 0.6486 | MCC: 0.0000 | ACC: 0.4800
Confusion matrix:
[[ 0 78]
 [ 0 72]]


-- Fold 2 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 329ms/step
[AAL][Fold 2] AUC: 0.4302 | F1: 0.3279 | MCC: -0.1250 | ACC: 0.4533
Confusion matrix:
[[48 34]
 [48 20]]


-- Fold 3 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 393ms/step
[AAL][Fold 3] AUC: 0.5378 | F1: 0.4394 | MCC: 0.0078 | ACC: 0.5067
Confusion matrix:
[[47 30]
 [44 29]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.50% | 0: 49.50% (n=1000)
Validation -> 1: 56.67% | 0: 43.33% (n=150)
Test -> 1: 58.00% | 0: 42.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 323ms/step
[BX][Fold 1] AUC: 0.4444 | F1: 0.6574 | MCC: -0.1487 | ACC: 0.5067
Confusion matrix:
[[ 5 58]
 [16 71]]


-- Fold 2 --
Train -> 1: 51.50% | 0: 48.50% (n=1000)
Validation -> 1: 58.00% | 0: 42.00% (n=150)
Test -> 1: 57.33% | 0: 42.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 319ms/step
[BX][Fold 2] AUC: 0.5071 | F1: 0.7265 | MCC: 0.0172 | ACC: 0.5733
Confusion matrix:
[[ 1 63]
 [ 1 85]]


-- Fold 3 --
Train -> 1: 53.30% | 0: 46.70% (n=1000)
Validation -> 1: 57.33% | 0: 42.67% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 330ms/step
[BX][Fold 3] AUC: 0.5934 | F1: 0.6941 | MCC: 0.0957 | ACC: 0.5533
Confusion matrix:
[[ 7 63]
 [ 4 76]]


-- Fold 4 --
Train -> 1: 55.20% | 0: 44.80% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3

...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 322ms/step
[VRTX][Fold 1] AUC: 0.4579 | F1: 0.7013 | MCC: 0.0000 | ACC: 0.5400
Confusion matrix:
[[ 0 69]
 [ 0 81]]


-- Fold 2 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 324ms/step
[VRTX][Fold 2] AUC: 0.4772 | F1: 0.6726 | MCC: 0.0000 | ACC: 0.5067
Confusion matrix:
[[ 0 74]
 [ 0 76]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 334ms/step
[VRTX][Fold 3] AUC: 0.4695 | F1: 0.6301 | MCC: 0.0000 | ACC: 0.4600
Confusion matrix:
[[ 0 81]
 [ 0 69]]


-- Fold 4 --
Train -> 1: 52.10% | 0: 47.90% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.70% | 0: 49.30% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 397ms/step
[WDC][Fold 1] AUC: 0.4993 | F1: 0.5731 | MCC: 0.0176 | ACC: 0.5133
Confusion matrix:
[[28 44]
 [29 49]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 325ms/step
[WDC][Fold 2] AUC: 0.5298 | F1: 0.5513 | MCC: 0.0808 | ACC: 0.5333
Confusion matrix:
[[37 44]
 [26 43]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 51.33% | 0: 48.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 328ms/step
[WDC][Fold 3] AUC: 0.5305 | F1: 0.6756 | MCC: 0.0031 | ACC: 0.5133
Confusion matrix:
[[ 1 72]
 [ 1 76]]


-- Fold 4 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 51.33% | 0: 48.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━

/tmp/ipython-input-3584323098.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 362ms/step
[WFC][Fold 1] AUC: 0.5290 | F1: 0.1053 | MCC: 0.1723 | ACC: 0.5467
Confusion matrix:
[[78  0]
 [68  4]]


-- Fold 2 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 350ms/step
[WFC][Fold 2] AUC: 0.4943 | F1: 0.0000 | MCC: 0.0000 | ACC: 0.4933
Confusion matrix:
[[74  0]
 [76  0]]


-- Fold 3 --
Train -> 1: 49.10% | 0: 50.90% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 345ms/step
[WFC][Fold 3] AUC: 0.4722 | F1: 0.6404 | MCC: -0.0302 | ACC: 0.5133
Confusion matrix:
[[12 57]
 [16 65]]


-- Fold 4 --
Train -> 1: 48.70% | 0: 51.30% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 10,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)

    # Return or use results_df
    # results_df  # now available for further processing


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 149ms/step
[AAL][Fold 1] AUC: 0.5037 | F1: 0.6256 | MCC: 0.0681 | ACC: 0.5133
Confusion matrix:
[[16 62]
 [11 61]]


-- Fold 2 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step
[AAL][Fold 2] AUC: 0.4711 | F1: 0.5172 | MCC: -0.0898 | ACC: 0.4400
Confusion matrix:
[[21 61]
 [23 45]]


-- Fold 3 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)


1/5 ━━━━━━━━━━━━━━━━━━━━ 2s 529ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step
[AAL][Fold 3] AUC: 0.4981 | F1: 0.5455 | MCC: 0.0062 | ACC: 0.5000
Confusion matrix:
[[30 47]
 [28 45]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 149ms/step
[AAL][Fold 4] AUC: 0.5123 | F1: 0.5000 | MCC: 0.0160 | ACC: 0.5067
Confusion matrix:
[[39 41]
 [33 37]]


-- Fold 5 --
Train -> 1: 47.40% | 0: 52.60% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 150ms/step
[AAL][Fold 5] AUC: 0.4605 | F1: 0.2045 | MCC: 0.0247 | ACC: 0.5333
Confusion matrix:
[[71  9]
 [61  9]]


==== Training for: BX ====


...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 144ms/step
[VRTX][Fold 1] AUC: 0.5199 | F1: 0.6933 | MCC: 0.0164 | ACC: 0.5400
Confusion matrix:
[[ 3 66]
 [ 3 78]]


-- Fold 2 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 145ms/step
[VRTX][Fold 2] AUC: 0.5030 | F1: 0.5325 | MCC: -0.0582 | ACC: 0.4733
Confusion matrix:
[[26 48]
 [31 45]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 150ms/step
[VRTX][Fold 3] AUC: 0.5523 | F1: 0.5763 | MCC: 0.0393 | ACC: 0.5000
Confusion matrix:
[[24 57]
 [18 51]]


-- Fold 4 --
Train -> 1: 52.10% | 0: 47.90% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.70% | 0: 49.30% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step
[WDC][Fold 1] AUC: 0.4829 | F1: 0.5067 | MCC: 0.0150 | ACC: 0.5067
Confusion matrix:
[[38 34]
 [40 38]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step
[WDC][Fold 2] AUC: 0.5888 | F1: 0.6012 | MCC: 0.1207 | ACC: 0.5400
Confusion matrix:
[[29 52]
 [17 52]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 51.33% | 0: 48.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 149ms/step
[WDC][Fold 3] AUC: 0.4915 | F1: 0.6604 | MCC: 0.0311 | ACC: 0.5200
Confusion matrix:
[[ 8 65]
 [ 7 70]]


-- Fold 4 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 51.33% | 0: 48.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step
[WFC][Fold 1] AUC: 0.3825 | F1: 0.3650 | MCC: -0.1670 | ACC: 0.4200
Confusion matrix:
[[38 40]
 [47 25]]


-- Fold 2 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step
[WFC][Fold 2] AUC: 0.6284 | F1: 0.0260 | MCC: 0.0808 | ACC: 0.5000
Confusion matrix:
[[74  0]
 [75  1]]


-- Fold 3 --
Train -> 1: 49.10% | 0: 50.90% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step
[WFC][Fold 3] AUC: 0.5065 | F1: 0.4430 | MCC: -0.1000 | ACC: 0.4467
Confusion matrix:
[[34 35]
 [48 33]]


-- Fold 4 --
Train -> 1: 48.70% | 0: 51.30% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 10,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 211ms/step
[AAL][Fold 1] AUC: 0.5372 | F1: 0.6445 | MCC: 0.0655 | ACC: 0.5000
Confusion matrix:
[[ 7 71]
 [ 4 68]]


-- Fold 2 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 210ms/step
[AAL][Fold 2] AUC: 0.5113 | F1: 0.5714 | MCC: -0.0289 | ACC: 0.4600
Confusion matrix:
[[15 67]
 [14 54]]


-- Fold 3 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 210ms/step
[AAL][Fold 3] AUC: 0.4729 | F1: 0.5422 | MCC: -0.0071 | ACC: 0.4933
Confusion matrix:
[[29 48]
 [28 45]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.50% | 0: 49.50% (n=1000)
Validation -> 1: 56.67% | 0: 43.33% (n=150)
Test -> 1: 58.00% | 0: 42.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 202ms/step
[BX][Fold 1] AUC: 0.4523 | F1: 0.6698 | MCC: -0.0855 | ACC: 0.5267
Confusion matrix:
[[ 7 56]
 [15 72]]


-- Fold 2 --
Train -> 1: 51.50% | 0: 48.50% (n=1000)
Validation -> 1: 58.00% | 0: 42.00% (n=150)
Test -> 1: 57.33% | 0: 42.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 202ms/step
[BX][Fold 2] AUC: 0.5013 | F1: 0.7249 | MCC: 0.0648 | ACC: 0.5800
Confusion matrix:
[[ 4 60]
 [ 3 83]]


-- Fold 3 --
Train -> 1: 53.30% | 0: 46.70% (n=1000)
Validation -> 1: 57.33% | 0: 42.67% (n=150)
Test -> 1: 53.33% | 0: 46.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 203ms/step
[BX][Fold 3] AUC: 0.5979 | F1: 0.7042 | MCC: 0.1714 | ACC: 0.5800
Confusion matrix:
[[12 58]
 [ 5 75]]


-- Fold 4 --
Train -> 1: 55.20% | 0: 44.80% (n=1000)
Validation -> 1: 53.33% | 0: 46.67% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2

...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 202ms/step
[VRTX][Fold 1] AUC: 0.4380 | F1: 0.6605 | MCC: -0.0589 | ACC: 0.5133
Confusion matrix:
[[ 6 63]
 [10 71]]


-- Fold 2 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 202ms/step
[VRTX][Fold 2] AUC: 0.5461 | F1: 0.6215 | MCC: 0.1088 | ACC: 0.5533
Confusion matrix:
[[28 46]
 [21 55]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 296ms/step
[VRTX][Fold 3] AUC: 0.4883 | F1: 0.5851 | MCC: 0.0086 | ACC: 0.4800
Confusion matrix:
[[17 64]
 [14 55]]


-- Fold 4 --
Train -> 1: 52.10% | 0: 47.90% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.70% | 0: 49.30% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 220ms/step
[WDC][Fold 1] AUC: 0.5244 | F1: 0.3130 | MCC: -0.0384 | ACC: 0.4733
Confusion matrix:
[[53 19]
 [60 18]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 216ms/step
[WDC][Fold 2] AUC: 0.5384 | F1: 0.5810 | MCC: 0.0423 | ACC: 0.5000
Confusion matrix:
[[23 58]
 [17 52]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 51.33% | 0: 48.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 268ms/step
[WDC][Fold 3] AUC: 0.5513 | F1: 0.6756 | MCC: 0.0031 | ACC: 0.5133
Confusion matrix:
[[ 1 72]
 [ 1 76]]


-- Fold 4 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 51.33% | 0: 48.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 207ms/step
[WFC][Fold 1] AUC: 0.4370 | F1: 0.2830 | MCC: -0.0421 | ACC: 0.4933
Confusion matrix:
[[59 19]
 [57 15]]


-- Fold 2 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 209ms/step
[WFC][Fold 2] AUC: 0.4520 | F1: 0.0952 | MCC: -0.0032 | ACC: 0.4933
Confusion matrix:
[[70  4]
 [72  4]]


-- Fold 3 --
Train -> 1: 49.10% | 0: 50.90% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 290ms/step
[WFC][Fold 3] AUC: 0.5429 | F1: 0.6257 | MCC: 0.0866 | ACC: 0.5533
Confusion matrix:
[[27 42]
 [25 56]]


-- Fold 4 --
Train -> 1: 48.70% | 0: 51.30% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━

In [ ]:
# ======================
# Config
# ======================
CONFIG = {
    "seed": 42,
    "window_size": 10,
    "price_cols": ['open', 'high', 'low', 'close', 'volume', 'DTWEXBGS', 'Price_x', 'WTI Crude Oil Price/Barrel'],
    "macro_cols": ['DFF', 'CPIAUCSL', 'UNRATE', 'PPIACO'],
    "sent_cols": ['avg_weighted_sent'],
    "target": 'target_binary',
    "feature_cols": None,
    "l2_value": 1e-4,
    "batch_size": 8,
    "epochs": 200,
    "learning_rate": 1e-3,
    "n_splits": 5,
    "early_stopping_patience": 5,
    "price_stack": [
        {"type": "conv1d", "filters": 16, "kernel_size": 3, "activation": "relu", "padding": "same"},
        {"type": "batchnorm"},
        {"type": "lstm", "units": 32, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 16, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
        ],
    "price_attention": True,
    "price_final_lstm_units": 16,
    "sent_stack": [
        {"type": "lstm", "units": 8, "recurrent_dropout": 0.25, "return_sequences": True},
        {"type": "dropout", "rate": 0.25},
    ],
    "sent_attention": True,
    "sent_final_lstm_units": 8,
}


# ======================
# Example run
# ======================
if __name__ == "__main__":
    try:
        merged
    except NameError:
        raise RuntimeError("Load `merged` DataFrame before running.")

    # Train and get results
    results_df = train_on_merged(merged, CONFIG)

    # Optionally summarize
    summarize_results(results_df)


==== Training for: AAL ====


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.60% | 0: 49.40% (n=1000)
Validation -> 1: 44.67% | 0: 55.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 302ms/step
[AAL][Fold 1] AUC: 0.5288 | F1: 0.6355 | MCC: -0.0095 | ACC: 0.4800
Confusion matrix:
[[ 4 74]
 [ 4 68]]


-- Fold 2 --
Train -> 1: 50.00% | 0: 50.00% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 45.33% | 0: 54.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 322ms/step
[AAL][Fold 2] AUC: 0.4414 | F1: 0.1837 | MCC: -0.1540 | ACC: 0.4667
Confusion matrix:
[[61 21]
 [59  9]]


-- Fold 3 --
Train -> 1: 49.00% | 0: 51.00% (n=1000)
Validation -> 1: 45.33% | 0: 54.67% (n=150)
Test -> 1: 48.67% | 0: 51.33% (n=150)


1/5 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 362ms/step
[AAL][Fold 3] AUC: 0.5296 | F1: 0.0506 | MCC: -0.0626 | ACC: 0.5000
Confusion matrix:
[[73  4]
 [71  2]]


-- Fold 4 --
Train -> 1: 48.30% | 0: 51.70% (n=1000)
Validation -> 1: 48.67% | 0: 51.33% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 311ms/step
[AAL][Fold 4] AUC: 0.5302 | F1: 0.1026 | MCC: 0.0159 | ACC: 0.5333
Confusion matrix:
[[76  4]
 [66  4]]


-- Fold 5 --
Train -> 1: 47.40% | 0: 52.60% (n=1000)
Validation -> 1: 46.67% | 0: 53.33% (n=150)
Test -> 1: 46.67% | 0: 53.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 456ms/step
[AAL][Fold 5] AUC: 0.4496 | F1: 0.0541 | MCC: 0.0111 | ACC: 0.5333
Confusion matrix:
[[78  2]
 [68  2]]


==== Training for: BX ====


...


-- Fold 1 --
Train -> 1: 51.60% | 0: 48.40% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 310ms/step
[VRTX][Fold 1] AUC: 0.4799 | F1: 0.7013 | MCC: 0.0000 | ACC: 0.5400
Confusion matrix:
[[ 0 69]
 [ 0 81]]


-- Fold 2 --
Train -> 1: 51.30% | 0: 48.70% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 304ms/step
[VRTX][Fold 2] AUC: 0.4787 | F1: 0.6726 | MCC: 0.0000 | ACC: 0.5067
Confusion matrix:
[[ 0 74]
 [ 0 76]]


-- Fold 3 --
Train -> 1: 51.80% | 0: 48.20% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 319ms/step
[VRTX][Fold 3] AUC: 0.4688 | F1: 0.6301 | MCC: 0.0000 | ACC: 0.4600
Confusion matrix:
[[ 0 81]
 [ 0 69]]


-- Fold 4 --
Train -> 1: 52.10% | 0: 47.90% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 50.70% | 0: 49.30% (n=1000)
Validation -> 1: 52.67% | 0: 47.33% (n=150)
Test -> 1: 52.00% | 0: 48.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 293ms/step
[WDC][Fold 1] AUC: 0.4847 | F1: 0.5333 | MCC: -0.0335 | ACC: 0.4867
Confusion matrix:
[[29 43]
 [34 44]]


-- Fold 2 --
Train -> 1: 51.70% | 0: 48.30% (n=1000)
Validation -> 1: 52.00% | 0: 48.00% (n=150)
Test -> 1: 46.00% | 0: 54.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 437ms/step
[WDC][Fold 2] AUC: 0.6005 | F1: 0.6092 | MCC: 0.1372 | ACC: 0.5467
Confusion matrix:
[[29 52]
 [16 53]]


-- Fold 3 --
Train -> 1: 52.30% | 0: 47.70% (n=1000)
Validation -> 1: 46.00% | 0: 54.00% (n=150)
Test -> 1: 51.33% | 0: 48.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 302ms/step
[WDC][Fold 3] AUC: 0.5410 | F1: 0.6794 | MCC: 0.1330 | ACC: 0.5533
Confusion matrix:
[[12 61]
 [ 6 71]]


-- Fold 4 --
Train -> 1: 51.10% | 0: 48.90% (n=1000)
Validation -> 1: 51.33% | 0: 48.67% (n=150)
Test -> 1: 49.33% | 0: 50.67% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipython-input-2124054096.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_symbol = df_symbol.fillna(method='ffill').fillna(method='bfill')



-- Fold 1 --
Train -> 1: 49.20% | 0: 50.80% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 48.00% | 0: 52.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 325ms/step
[WFC][Fold 1] AUC: 0.4854 | F1: 0.2545 | MCC: -0.1301 | ACC: 0.4533
Confusion matrix:
[[54 24]
 [58 14]]


-- Fold 2 --
Train -> 1: 48.90% | 0: 51.10% (n=1000)
Validation -> 1: 48.00% | 0: 52.00% (n=150)
Test -> 1: 50.67% | 0: 49.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 309ms/step
[WFC][Fold 2] AUC: 0.4589 | F1: 0.2524 | MCC: -0.0236 | ACC: 0.4867
Confusion matrix:
[[60 14]
 [63 13]]


-- Fold 3 --
Train -> 1: 49.10% | 0: 50.90% (n=1000)
Validation -> 1: 50.67% | 0: 49.33% (n=150)
Test -> 1: 54.00% | 0: 46.00% (n=150)
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 309ms/step
[WFC][Fold 3] AUC: 0.4863 | F1: 0.6154 | MCC: -0.0489 | ACC: 0.5000
Confusion matrix:
[[15 54]
 [21 60]]


-- Fold 4 --
Train -> 1: 48.70% | 0: 51.30% (n=1000)
Validation -> 1: 54.00% | 0: 46.00% (n=150)
Test -> 1: 54.67% | 0: 45.33% (n=150)
5/5 ━━━━━━━━━━━━━━━━━